# Adaptive NIDS: IDS2025 baseline and cross-dataset evaluation

This notebook separates three different questions:

1. IDS2025 in-dataset baseline: known multiclass recognition and a held-out DoS/DDoS unknown-attack experiment.
2. Cross-dataset test: train and calibrate using IDS2025 only, then evaluate on held-out CSE-CIC-IDS2018 rows using a shared feature schema.
3. Optional target adaptation / pooled training: after the zero-shot external result is recorded, use a separate analyst-labeled CSE-CIC-IDS2018 batch, retrain a candidate, validate it, and test it on untouched external rows.

Adding CSE-CIC-IDS2018 to training may help target performance, but it is then a pooled or adapted model. It is no longer a pure cross-dataset test. Results may still be poor because traffic distributions and labels differ. This notebook samples CSE-CIC-IDS2018 per file to fit within typical Colab memory; record the sampling limits with all results.

The autoencoder is fitted only on Normal training flows. Its threshold uses validation flows, and it is evaluated separately from class recognition. Concept drift is not implemented here.

## 1. Setup and imports

Run this notebook in Google Colab. Install the extra model library, then import the data and evaluation tools.

In [ ]:
%pip -q install xgboost seaborn

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import mutual_info_classif
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, RobustScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    recall_score, average_precision_score
)
from sklearn.base import clone
from xgboost import XGBClassifier

SEED = 42
np.random.seed(SEED)

## 2. Connect Drive and set dataset locations

Extract the CSE-CIC-IDS2018 ZIP into its own Drive folder. Keep IDS2025 and CSE-CIC-IDS2018 in separate folders so the notebook can control which dataset is used for training and which is used only for external testing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

IDS2025_CSV = Path('/content/drive/MyDrive/NIDS/IDS2025.csv')
IDS2018_DIR = Path('/content/drive/MyDrive/NIDS/CSE-CIC-IDS2018')

SAMPLE_PER_2018_FILE = 10_000  # Raise after checking Colab memory and runtime.
CHUNK_SIZE = 100_000

assert IDS2025_CSV.exists(), f'IDS2025 file not found: {IDS2025_CSV}'
assert IDS2018_DIR.exists(), f'IDS2018 folder not found: {IDS2018_DIR}'
ids2018_files = sorted(IDS2018_DIR.rglob('*.csv'))
print('CSE-CIC-IDS2018 CSV files:', len(ids2018_files))
print('\n'.join(str(p) for p in ids2018_files[:20]))
assert ids2018_files, 'Extract the ZIP and set IDS2018_DIR to its CSV folder.'

Mounted at /content/drive
CSE-CIC-IDS2018 CSV files: 14
/content/drive/MyDrive/NIDS/CSE-CIC-IDS2018/Bot.csv
/content/drive/MyDrive/NIDS/CSE-CIC-IDS2018/Brute Force -Web.csv
/content/drive/MyDrive/NIDS/CSE-CIC-IDS2018/Brute Force -XSS.csv
/content/drive/MyDrive/NIDS/CSE-CIC-IDS2018/DDOS attack-HOIC.csv
/content/drive/MyDrive/NIDS/CSE-CIC-IDS2018/DDOS attack-LOIC-UDP.csv
/content/drive/MyDrive/NIDS/CSE-CIC-IDS2018/DDoS attacks-LOIC-HTTP.csv
/content/drive/MyDrive/NIDS/CSE-CIC-IDS2018/DoS attacks-GoldenEye.csv
/content/drive/MyDrive/NIDS/CSE-CIC-IDS2018/DoS attacks-Hulk.csv
/content/drive/MyDrive/NIDS/CSE-CIC-IDS2018/DoS attacks-SlowHTTPTest.csv
/content/drive/MyDrive/NIDS/CSE-CIC-IDS2018/DoS attacks-Slowloris.csv
/content/drive/MyDrive/NIDS/CSE-CIC-IDS2018/FTP-BruteForce.csv
/content/drive/MyDrive/NIDS/CSE-CIC-IDS2018/Infilteration.csv
/content/drive/MyDrive/NIDS/CSE-CIC-IDS2018/SQL Injection.csv
/content/drive/MyDrive/NIDS/CSE-CIC-IDS2018/SSH-Bruteforce.csv


## 3. Standardize feature names and labels

IDS2025 uses newLabel; CSE-CIC-IDS2018 uses Label. The two files also use different abbreviations for many CICFlowMeter features. These mappings align common names. We exclude source-file names from model inputs and use row labels rather than inferring labels from filenames.

In [ ]:
def compact_name(name):
    return re.sub(r'[^a-z0-9]+', '', str(name).lower())

ALIASES = {
    'newlabel': 'label',
    'dstport': 'destinationport',
    'totfwdpkts': 'totalfwdpackets',
    'totbwdpkts': 'totalbackwardpackets',
    'totlenfwdpkts': 'totallengthoffwdpackets',
    'totlenbwdpkts': 'totallengthofbwdpackets',
    'fwdpktlenmax': 'fwdpacketlengthmax', 'fwdpktlenmin': 'fwdpacketlengthmin',
    'fwdpktlenmean': 'fwdpacketlengthmean', 'fwdpktlenavg': 'fwdpacketlengthmean',
    'fwdpktlenstd': 'fwdpacketlengthstd',
    'bwdpktlenmax': 'bwdpacketlengthmax', 'bwdpktlenmin': 'bwdpacketlengthmin',
    'bwdpktlenmean': 'bwdpacketlengthmean', 'bwdpktlenavg': 'bwdpacketlengthmean',
    'bwdpktlenstd': 'bwdpacketlengthstd',
    'flowbytss': 'flowbytess', 'flowpktss': 'flowpacketss',
    'fwdiattot': 'fwdiattotal', 'bwdiattot': 'bwdiattotal',
    'fwdheaderlen': 'fwdheaderlength', 'bwdheaderlen': 'bwdheaderlength',
    'fwdpktss': 'fwdpacketss', 'bwdpktss': 'bwdpacketss',
    'pktlenmin': 'minpacketlength', 'pktlenmax': 'maxpacketlength',
    'pktlenmean': 'packetlengthmean', 'pktlenavg': 'packetlengthmean',
    'pktlenstd': 'packetlengthstd', 'pktlenvar': 'packetlengthvariance',
    'finflagcnt': 'finflagcount', 'synflagcnt': 'synflagcount',
    'rstflagcnt': 'rstflagcount', 'pshflagcnt': 'pshflagcount',
    'ackflagcnt': 'ackflagcount', 'urgflagcnt': 'urgflagcount',
    'eceflagcnt': 'eceflagcount',
    'pktsizeavg': 'averagepacketsize',
    'fwdsegsizeavg': 'avgfwdsegmentsize', 'bwdsegsizeavg': 'avgbwdsegmentsize',
    'fwdbytsbavg': 'fwdavgbytesbulk', 'fwdpktsbavg': 'fwdavgpacketsbulk',
    'fwdblkrateavg': 'fwdavgbulkrate',
    'bwdbytsbavg': 'bwdavgbytesbulk', 'bwdpktsbavg': 'bwdavgpacketsbulk',
    'bwdblkrateavg': 'bwdavgbulkrate',
    'subflowfwdpkts': 'subflowfwdpackets', 'subflowbwdpkts': 'subflowbackwardpackets',
    'subflowfwdb yts': 'subflowfwdbytes', 'subflowfwdbyts': 'subflowfwdbytes',
    'subflowbwdb yts': 'subflowbackwardbytes', 'subflowbwdbyts': 'subflowbackwardbytes',
    'initfwdwinbyts': 'initwinbytesforward',
    'initbwdwinbyts': 'initwinbytesbackward',
    'fwdactdatapkts': 'actdatapktfwd',
    'fwdsegsizemin': 'minsegsizeforward',
}

def canonical_column(name):
    key = compact_name(name)
    return ALIASES.get(key, key)

def standardize_columns(frame):
    frame = frame.copy()
    frame.columns = [canonical_column(c) for c in frame.columns]
    return frame.loc[:, ~pd.Index(frame.columns).duplicated()]

def common_label(value):
    s = str(value).strip().lower().replace('_', ' ').replace('-', ' ')
    if s in {'normal', 'benign'}:
        return 'Normal'
    if 'infiltrat' in s:
        return 'Infiltration'
    if 'bot' in s:
        return 'Botnet'
    if 'heartbleed' in s:
        return 'Heartbleed'
    # Check web subtypes before the generic brute-force condition.
    if any(x in s for x in ['xss', 'sql injection', 'web attack', 'web']):
        return 'Web Attack'
    if any(x in s for x in ['brute', 'patator', 'ftp', 'ssh']):
        return 'Brute Force'
    if 'ddos' in s or s.startswith('dos ') or s == 'dos':
        return 'DoS/DDoS'
    if 'portscan' in s or 'port scan' in s:
        return 'PortScan'
    return 'Other: ' + s

## 4. Load IDS2025 and sample CSE-CIC-IDS2018 safely

The CSE-CIC-IDS2018 ZIP contains multiple CSVs. The sampler uses a streaming uniform reservoir sample per CSV rather than taking only the first rows, which could overrepresent early traffic. Increase the per-file cap for a larger run and record the value in your report.

In [ ]:
def clean_loaded_frame(frame, dataset_name, source_name):
    frame = standardize_columns(frame)
    if 'label' not in frame.columns:
        raise ValueError(f'No target label column found in {source_name}: {frame.columns.tolist()}')
    raw_label = frame['label'].astype(str).str.strip()
    frame = frame[raw_label.str.lower().ne('label') & raw_label.str.lower().ne('nan')].copy()
    frame['Label'] = raw_label.loc[frame.index].map(common_label)
    frame['_source_dataset'] = dataset_name
    frame['_source_file'] = source_name
    return frame

ids2025 = pd.read_csv(IDS2025_CSV, low_memory=False, encoding_errors='replace')
ids2025 = clean_loaded_frame(ids2025, 'IDS2025', IDS2025_CSV.name)
print('IDS2025 shape:', ids2025.shape)
print(ids2025['Label'].value_counts(dropna=False))

def reservoir_sample_csv(path, limit, seed=SEED, chunksize=CHUNK_SIZE):
    rng = np.random.default_rng(seed)
    reservoir = None
    for chunk in pd.read_csv(path, chunksize=chunksize, low_memory=False,
                             encoding_errors='replace'):
        chunk = clean_loaded_frame(chunk, 'CSE-CIC-IDS2018', path.name)
        chunk['_sample_key'] = rng.random(len(chunk))
        if reservoir is None:
            reservoir = chunk
        else:
            reservoir = pd.concat([reservoir, chunk], ignore_index=True)
        if len(reservoir) > limit:
            reservoir = reservoir.nlargest(limit, '_sample_key').reset_index(drop=True)
    if reservoir is None:
        return pd.DataFrame()
    return reservoir.drop(columns=['_sample_key']).reset_index(drop=True)

parts = []
for i, path in enumerate(ids2018_files):
    sampled = reservoir_sample_csv(path, SAMPLE_PER_2018_FILE, seed=SEED + i)
    if len(sampled):
        parts.append(sampled)
        print(path.name, 'sampled rows:', len(sampled))

ids2018 = pd.concat(parts, ignore_index=True)
print('CSE-CIC-IDS2018 sampled shape:', ids2018.shape)
print(ids2018['Label'].value_counts(dropna=False))

IDS2025 shape: (91830, 83)
Label
Normal          26185
DoS/DDoS        26066
PortScan        25409
Brute Force     10201
Web Attack       2067
Botnet           1873
Infiltration       29
Name: count, dtype: int64
Bot.csv sampled rows: 10000
Brute Force -Web.csv sampled rows: 2073
Brute Force -XSS.csv sampled rows: 734
DDOS attack-HOIC.csv sampled rows: 10000
DDOS attack-LOIC-UDP.csv sampled rows: 5784
DDoS attacks-LOIC-HTTP.csv sampled rows: 10000
DoS attacks-GoldenEye.csv sampled rows: 10000
DoS attacks-Hulk.csv sampled rows: 10000
DoS attacks-SlowHTTPTest.csv sampled rows: 10000
DoS attacks-Slowloris.csv sampled rows: 10000
FTP-BruteForce.csv sampled rows: 10000
Infilteration.csv sampled rows: 10000
SQL Injection.csv sampled rows: 286
SSH-Bruteforce.csv sampled rows: 10000
CSE-CIC-IDS2018 sampled shape: (108877, 82)
Label
Normal                  76918
DoS/DDoS                19280
Brute Force              6024
Other: infilteration     2961
Botnet                   2766
Web Attack    

## 5. Audit labels and align features

Use the row-level Label field, not a scenario filename. Review label support before modeling. IDS2025 merges some attack variants, while CSE-CIC-IDS2018 has detailed labels. The shared DoS/DDoS category therefore cannot isolate DDoS alone.

In [ ]:
print('IDS2025 labels')
display(ids2025['Label'].value_counts().to_frame('rows'))
print('CSE-CIC-IDS2018 labels')
display(ids2018['Label'].value_counts().to_frame('sampled_rows'))

META = {'label', 'Label', '_source_dataset', '_source_file'}
f2025 = set(ids2025.columns) - META
f2018 = set(ids2018.columns) - META
COMMON_FEATURES = sorted(f2025 & f2018)
print('Common features:', len(COMMON_FEATURES))
print('IDS2025-only features:', sorted(f2025 - f2018))
print('IDS2018-only features:', sorted(f2018 - f2025))
assert len(COMMON_FEATURES) >= 40, 'Too few aligned features; inspect aliases before modeling.'

# Keep only the same feature set in both domains. Coerce features to numeric.
for frame in (ids2025, ids2018):
    for col in COMMON_FEATURES:
        frame[col] = pd.to_numeric(frame[col], errors='coerce')
    frame[COMMON_FEATURES] = frame[COMMON_FEATURES].replace([np.inf, -np.inf], np.nan)

# Remove exact duplicates within each dataset using aligned features and labels.
ids2025 = ids2025.drop_duplicates(subset=COMMON_FEATURES + ['Label']).reset_index(drop=True)
ids2018 = ids2018.drop_duplicates(subset=COMMON_FEATURES + ['Label']).reset_index(drop=True)

print('IDS2025 post-audit:', ids2025.shape)
print('IDS2018 post-audit sample:', ids2018.shape)

IDS2025 labels


,rows
Label,
Normal,26185
DoS/DDoS,26066
PortScan,25409
Brute Force,10201
Web Attack,2067
Botnet,1873
Infiltration,29


CSE-CIC-IDS2018 labels


,sampled_rows
Label,
Normal,76918
DoS/DDoS,19280
Brute Force,6024
Other: infilteration,2961
Botnet,2766
Web Attack,928


Common features: 76
IDS2025-only features: ['sourceport', 'subflowbwdbytes', 'subflowbwdpackets']
IDS2018-only features: ['subflowbackwardbytes', 'subflowbackwardpackets']
IDS2025 post-audit: (80116, 83)
IDS2018 post-audit sample: (96461, 82)


## 6. Create source-domain train, validation, and test sets

The main unknown class is DoS/DDoS, held out from IDS2025 training. For the initial in-domain experiment, reserve a labeled IDS2025 subset for unknown evaluation. The source-model test set combines known-class test rows with the held-out attack test rows. Keep separate CSE-CIC-IDS2018 data for the cross-dataset experiment.

In [ ]:
UNKNOWN_LABELS = {'DoS/DDoS'}
missing = UNKNOWN_LABELS - set(ids2025['Label'].unique())
assert not missing, f'Unknown labels are not present: {missing}'

known_df = ids2025[~ids2025['Label'].isin(UNKNOWN_LABELS)].copy()
unknown_df = ids2025[ids2025['Label'].isin(UNKNOWN_LABELS)].copy()
unknown_test_2025 = unknown_df.copy()

train_val_df, test_known_df = train_test_split(
    known_df, test_size=0.20, random_state=SEED, stratify=known_df['Label']
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.20, random_state=SEED, stratify=train_val_df['Label']
)

X_train = train_df[COMMON_FEATURES].copy()
X_val = val_df[COMMON_FEATURES].copy()
X_test_known = test_known_df[COMMON_FEATURES].copy()
X_test_unknown = unknown_test_2025[COMMON_FEATURES].copy()
y_train = train_df['Label'].reset_index(drop=True)
y_val = val_df['Label'].reset_index(drop=True)
y_test_known = test_known_df['Label'].reset_index(drop=True)

print('Train / validation / known test:', len(X_train), len(X_val), len(X_test_known))
print('Held-out DoS/DDoS test:', len(X_test_unknown))
print('Low-support classes in train:')
display(y_train.value_counts().to_frame('train_rows'))

Train / validation / known test: 37599 9400 11750
Held-out DoS/DDoS test: 21367
Low-support classes in train:


,train_rows
Label,
Normal,16550
PortScan,14130
Brute Force,4410
Web Attack,1300
Botnet,1190
Infiltration,19


## 7. Feature reduction using source training data only

The previous notebook computed a full-dataset correlation matrix but did not remove features. This step ranks source-training features by mutual information with the known training labels, then removes one of any pair with absolute Spearman correlation at or above 0.98. The higher-ranked feature is kept. Validation and test labels are not used to choose the feature set.

In [ ]:
numeric_features = X_train.select_dtypes(include=np.number).columns.tolist()
Xn = X_train[numeric_features].replace([np.inf, -np.inf], np.nan)
nonconstant = Xn.columns[Xn.nunique(dropna=True) > 1].tolist()
Xn = Xn[nonconstant]

imputer_fs = SimpleImputer(strategy='median')
Xn_imp = imputer_fs.fit_transform(Xn)
mi = mutual_info_classif(Xn_imp, y_train, random_state=SEED)
mi_scores = pd.Series(mi, index=nonconstant).sort_values(ascending=False)
corr_abs = pd.DataFrame(Xn_imp, columns=nonconstant).corr(method='spearman').abs()

selected_numeric = []
for feature in mi_scores.index:
    if not any(corr_abs.loc[feature, kept] >= 0.98 for kept in selected_numeric):
        selected_numeric.append(feature)

MODEL_FEATURES = selected_numeric
removed_features = [c for c in COMMON_FEATURES if c not in MODEL_FEATURES]
print(f'Features before: {len(COMMON_FEATURES)}')
print(f'Features retained: {len(MODEL_FEATURES)}')
print(f'Features removed: {len(removed_features)}')
print('Removed correlated/constant features:', removed_features)
display(mi_scores.head(20).to_frame('training_mutual_information'))

# Apply the fixed source-selected list everywhere. Never select using CSE test labels.
X_train = X_train[MODEL_FEATURES].copy()
X_val = X_val[MODEL_FEATURES].copy()
X_test_known = X_test_known[MODEL_FEATURES].copy()
X_test_unknown = X_test_unknown[MODEL_FEATURES].copy()
X2018 = ids2018[MODEL_FEATURES].copy()

Features before: 76
Features retained: 42
Features removed: 34
Removed correlated/constant features: ['activemean', 'activemin', 'averagepacketsize', 'bwdavgbulkrate', 'bwdavgbytesbulk', 'bwdavgpacketsbulk', 'bwdiatmean', 'bwdiattotal', 'bwdpacketlengthmax', 'bwdpacketlengthmean', 'bwdpshflags', 'bwdurgflags', 'cweflagcount', 'flowduration', 'flowiatmax', 'flowiatmean', 'fwdavgbulkrate', 'fwdavgbytesbulk', 'fwdavgpacketsbulk', 'fwdiatmean', 'fwdiattotal', 'fwdpacketlengthmax', 'fwdpacketlengthmean', 'fwdpacketss', 'fwdpshflags', 'fwdurgflags', 'idlemax', 'idlemean', 'idlemin', 'minpacketlength', 'packetlengthvariance', 'rstflagcount', 'subflowfwdpackets', 'totallengthoffwdpackets']


,training_mutual_information
packetlengthmean,1.046334
packetlengthstd,1.030384
packetlengthvariance,1.029789
destinationport,1.012488
averagepacketsize,1.000935
initwinbytesbackward,0.933747
flowbytess,0.875715
totallengthofbwdpackets,0.866018
subflowfwdbytes,0.857144
totallengthoffwdpackets,0.856239


## 8. Train known-class Random Forest and XGBoost models

Both are multiclass classifiers trained on the same known IDS2025 classes. A validation confidence threshold flags lower-confidence rows as Unknown/Suspicious. The threshold is calibrated on the source validation set and then frozen for the IDS2018 test.

In [ ]:
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
preprocess = ColumnTransformer(
    [('num', SimpleImputer(strategy='median'), num_cols)],
    remainder='drop'
)

rf = Pipeline([
    ('preprocess', clone(preprocess)),
    ('model', RandomForestClassifier(
        n_estimators=400,
        class_weight='balanced_subsample',
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=SEED,
    )),
])
rf.fit(X_train, y_train)
rf_val_probs = rf.predict_proba(X_val)
rf_threshold = np.quantile(rf_val_probs.max(axis=1), 0.05)
print('RF source-validation confidence threshold:', rf_threshold)

label_encoder = LabelEncoder()
y_train_id = label_encoder.fit_transform(y_train)
xgb = Pipeline([
    ('preprocess', clone(preprocess)),
    ('model', XGBClassifier(
        objective='multi:softprob',
        num_class=len(label_encoder.classes_),
        n_estimators=500,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric='mlogloss',
        tree_method='hist',
        n_jobs=-1,
        random_state=SEED,
    )),
])
xgb.fit(X_train, y_train_id)
xgb_val_probs = xgb.predict_proba(X_val)
xgb_threshold = np.quantile(xgb_val_probs.max(axis=1), 0.05)
print('XGB source-validation confidence threshold:', xgb_threshold)

RF source-validation confidence threshold: 0.9534316591987837
XGB source-validation confidence threshold: 0.9988195


## 9. Evaluate the IDS2025 open-set baseline

Report known-class performance separately from unknown detection. Macro-F1 and per-class recall are important because Infiltration has very few rows. Do not rely on accuracy alone.

In [ ]:
X_test = pd.concat([X_test_known, X_test_unknown], ignore_index=True)
y_test = pd.concat([y_test_known, unknown_test_2025['Label'].reset_index(drop=True)], ignore_index=True)
known_mask = ~y_test.isin(UNKNOWN_LABELS).to_numpy()
unknown_mask = y_test.isin(UNKNOWN_LABELS).to_numpy()

def predict_with_rejection(model, X, threshold, classes=None, encoder=None):
    probs = model.predict_proba(X)
    if encoder is not None:
        labels = encoder.inverse_transform(probs.argmax(axis=1)).astype(object)
    else:
        labels = np.asarray(model.classes_)[probs.argmax(axis=1)].astype(object)
    labels[probs.max(axis=1) < threshold] = 'Unknown/Suspicious'
    return labels

rf_pred = predict_with_rejection(rf, X_test, rf_threshold)
xgb_pred = predict_with_rejection(xgb, X_test, xgb_threshold, encoder=label_encoder)

for name, pred in [('Random Forest', rf_pred), ('XGBoost', xgb_pred)]:
    print('\n', name, '- known-class report')
    print(classification_report(y_test[known_mask], pred[known_mask], zero_division=0))
    rejected = pred == 'Unknown/Suspicious'
    print('Held-out DoS/DDoS flagged unknown:', rejected[unknown_mask].mean())
    print('Known rows rejected as unknown:', rejected[known_mask].mean())


 Random Forest - known-class report
                    precision    recall  f1-score   support

            Botnet       0.99      0.91      0.95       372
       Brute Force       1.00      0.96      0.98      1378
      Infiltration       1.00      0.17      0.29         6
            Normal       1.00      0.92      0.96      5172
          PortScan       1.00      0.99      1.00      4416
Unknown/Suspicious       0.00      0.00      0.00         0
        Web Attack       1.00      0.89      0.94       406

          accuracy                           0.95     11750
         macro avg       0.85      0.69      0.73     11750
      weighted avg       1.00      0.95      0.97     11750

Held-out DoS/DDoS flagged unknown: 0.6324706322834277
Known rows rejected as unknown: 0.04902127659574468

 XGBoost - known-class report
                    precision    recall  f1-score   support

            Botnet       1.00      0.71      0.83       372
       Brute Force       1.00      0.98   

## 10. Benign-only autoencoder

Train the autoencoder on Normal flows from the same source training split. The validation data includes Normal and known attack classes, but only validation scores are used to calibrate a threshold under a maximum 5% Normal false-alarm rate. DoS/DDoS remains unseen during threshold selection.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(SEED)
normal_train = y_train.eq('Normal').to_numpy()
normal_val = y_val.eq('Normal').to_numpy()

# Robust imputation and scaling are fit using Normal training flows only.
Xae_train_raw = X_train.loc[normal_train].copy()
Xae_val_raw = X_val.copy()
Xae_test_raw = X_test.copy()
ae_medians = Xae_train_raw.median()
Xae_train = Xae_train_raw.fillna(ae_medians)
Xae_val = Xae_val_raw.fillna(ae_medians)
Xae_test = Xae_test_raw.fillna(ae_medians)
ae_scaler = RobustScaler()
Xae_train_s = np.clip(ae_scaler.fit_transform(Xae_train), -10, 10)
Xae_val_s = np.clip(ae_scaler.transform(Xae_val), -10, 10)
Xae_test_s = np.clip(ae_scaler.transform(Xae_test), -10, 10)

input_dim = Xae_train_s.shape[1]
inputs = keras.Input(shape=(input_dim,))
x = layers.Dense(64, activation='relu')(inputs)
x = layers.Dense(32, activation='relu')(x)
latent = layers.Dense(16, activation='relu')(x)
x = layers.Dense(32, activation='relu')(latent)
x = layers.Dense(64, activation='relu')(x)
outputs = layers.Dense(input_dim, activation='linear')(x)
autoencoder = keras.Model(inputs, outputs)
autoencoder.compile(optimizer='adam', loss='mse')
early_stop = keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
history = autoencoder.fit(
    Xae_train_s, Xae_train_s,
    validation_data=(Xae_val_s[normal_val], Xae_val_s[normal_val]),
    epochs=100, batch_size=256, shuffle=True,
    callbacks=[early_stop], verbose=1,
)

Epoch 1/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - loss: 5.0154 - val_loss: 2.9210
Epoch 2/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 2.0668 - val_loss: 1.4448
Epoch 3/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 1.1829 - val_loss: 1.0038
Epoch 4/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.8372 - val_loss: 0.7111
Epoch 5/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.5931 - val_loss: 0.5279
Epoch 6/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0.4581 - val_loss: 0.4226
Epoch 7/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3788 - val_loss: 0.3606
Epoch 8/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.3300 - val_loss: 0.3191
Epoch 9/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.2948 - val_loss: 0.2912
Epoch 10/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.2686 - val_loss: 0.2682
Epoch 11/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - loss: 0.2477 - val_loss: 0.2489
Epoch 12/100
65/65 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 0

In [ ]:
def reconstruction_errors(model, scaled):
    reconstructed = model.predict(scaled, verbose=0)
    return np.mean(np.square(scaled - reconstructed), axis=1)

val_errors = reconstruction_errors(autoencoder, Xae_val_s)
val_is_attack = y_val.ne('Normal').to_numpy()
val_is_normal = ~val_is_attack
threshold_rows = []
for t in np.unique(np.quantile(val_errors, np.linspace(0, 1, 501))):
    pred = val_errors >= t
    normal_fpr = pred[val_is_normal].mean()
    if normal_fpr <= 0.05:
        threshold_rows.append((f1_score(val_is_attack, pred), t, normal_fpr))
_, ae_threshold, ae_val_fpr = max(threshold_rows, key=lambda x: x[0])
print('AE threshold:', ae_threshold)
print('Validation Normal false-alarm rate:', f'{ae_val_fpr:.2%}')

ae_test_errors = reconstruction_errors(autoencoder, Xae_test_s)
ae_pred = ae_test_errors >= ae_threshold
y_test_attack = y_test.ne('Normal').to_numpy()
print(classification_report(y_test_attack, ae_pred, target_names=['Normal', 'Attack'], zero_division=0))
print('Test confusion matrix [Normal, Attack]:')
print(confusion_matrix(y_test_attack, ae_pred.astype(int)))
print('Held-out DoS/DDoS anomaly recall:', ae_pred[unknown_mask].mean())
print('Normal test false-alarm rate:', ae_pred[y_test.eq('Normal').to_numpy()].mean())
print('Attack prevalence / AP:', y_test_attack.mean(), average_precision_score(y_test_attack, ae_test_errors))

AE threshold: 0.34740883415193474
Validation Normal false-alarm rate: 4.86%
              precision    recall  f1-score   support

      Normal       0.30      0.95      0.46      5172
      Attack       0.99      0.59      0.74     27945

    accuracy                           0.65     33117
   macro avg       0.64      0.77      0.60     33117
weighted avg       0.88      0.65      0.69     33117

Test confusion matrix [Normal, Attack]:
[[ 4939   233]
 [11484 16461]]
Held-out DoS/DDoS anomaly recall: 0.7348247297234053
Normal test false-alarm rate: 0.04505027068832173
Attack prevalence / AP: 0.843826433553764 0.9681644901468562


## 11. Cross-dataset experiment: train IDS2025, test CSE-CIC-IDS2018

The CSE-CIC-IDS2018 sample is split by label into a target-domain training pool and an untouched external test set. The zero-shot model uses no CSE rows. For the later adaptation demonstration, only reviewed DoS/DDoS rows from the separate CSE training pool are used. Other CSE training-pool rows are reserved for the optional pooled-training comparison. Results describe this sampled ZIP version and sample cap.

In [ ]:
# Split each target label separately: keep 20% as a labeled target-domain pool
# and reserve 80% as the untouched external test set.
cse_pool_parts = []
cse_test_parts = []
for label, group in ids2018.groupby('Label'):
    if len(group) < 5:
        # Very small classes cannot be split reliably; preserve them only for external testing.
        cse_test_parts.append(group)
    else:
        pool_part, test_part = train_test_split(
            group, test_size=0.80, random_state=SEED
        )
        cse_pool_parts.append(pool_part)
        cse_test_parts.append(test_part)

cse_train_pool = pd.concat(cse_pool_parts, ignore_index=True)
cse_external_test = pd.concat(cse_test_parts, ignore_index=True)
cse_dos_pool = cse_train_pool[cse_train_pool['Label'].eq('DoS/DDoS')].copy()
assert len(cse_dos_pool) >= 2, 'Not enough target DoS/DDoS rows for adaptation.'
cse_adapt_train_df, cse_adapt_val_df = train_test_split(
    cse_dos_pool, test_size=0.20, random_state=SEED
)
# Simulate a limited analyst-reviewed batch.
N_ADAPT = min(1000, len(cse_adapt_train_df))
cse_adapt_train_df = cse_adapt_train_df.sample(n=N_ADAPT, random_state=SEED)

X_cse_test = cse_external_test[MODEL_FEATURES].copy()
y_cse_test = cse_external_test['Label'].reset_index(drop=True)
cse_known = y_cse_test.isin(rf.classes_).to_numpy()
cse_unknown = ~cse_known
print('CSE target-domain training pool:', len(cse_train_pool))
print('CSE untouched external test rows:', len(cse_external_test))
print('CSE adaptation batch:', len(cse_adapt_train_df))
print('CSE adaptation validation rows:', len(cse_adapt_val_df))
print('External test class counts:')
display(y_cse_test.value_counts().to_frame('test_rows'))

CSE target-domain training pool: 19290
CSE untouched external test rows: 77171
CSE adaptation batch: 1000
CSE adaptation validation rows: 629
External test class counts:


,test_rows
Label,
Normal,58833
DoS/DDoS,12566
Other: infilteration,2347
Botnet,1406
Brute Force,1330
Web Attack,689


In [ ]:
# Zero-shot cross-dataset test:
# The models and thresholds were fitted using IDS2025 only.

for name, model, threshold, encoder in [
    ("Random Forest", rf, rf_threshold, None),
    ("XGBoost", xgb, xgb_threshold, label_encoder),
]:
    pred = predict_with_rejection(
        model,
        X_cse_test,
        threshold,
        encoder=encoder,
    )

    rejected = pred == "Unknown/Suspicious"

    print(f"\n{name} — CSE-CIC-IDS2018 results")
    print("Test rows:", len(y_cse_test))
    print("Known-label rows:", int(cse_known.sum()))
    print("Unseen-label rows:", int(cse_unknown.sum()))

    if cse_known.any():
        print("\nKnown-class performance:")
        print(classification_report(
            y_cse_test[cse_known],
            pred[cse_known],
            zero_division=0,
        ))

    if cse_unknown.any():
        print("\nUnseen-class detection:")
        print("Unseen labels:", sorted(y_cse_test[cse_unknown].unique()))
        print("Unseen-label recall:",
              rejected[cse_unknown].mean())
        print("Known rows incorrectly flagged unknown:",
              rejected[cse_known].mean() if cse_known.any() else float("nan"))


# Test the autoencoder using its IDS2025-trained scaler and threshold.
X_cse_ae = X_cse_test.fillna(ae_medians)
X_cse_ae_scaled = np.clip(
    ae_scaler.transform(X_cse_ae),
    -10,
    10,
)

cse_ae_errors = reconstruction_errors(autoencoder, X_cse_ae_scaled)
cse_ae_flagged = cse_ae_errors >= ae_threshold
cse_is_attack = y_cse_test.ne("Normal").to_numpy()

print("\nAutoencoder — CSE-CIC-IDS2018 anomaly results")
print(classification_report(
    cse_is_attack,
    cse_ae_flagged,
    target_names=["Normal", "Attack"],
    zero_division=0,
))
print("Normal false-alarm rate:",
      cse_ae_flagged[y_cse_test.eq("Normal").to_numpy()].mean())

if cse_unknown.any():
    print("Unseen-label anomaly recall:",
          cse_ae_flagged[cse_unknown].mean())


Random Forest — CSE-CIC-IDS2018 results
Test rows: 77171
Known-label rows: 62258
Unseen-label rows: 14913

Known-class performance:
                    precision    recall  f1-score   support

            Botnet       0.00      0.00      0.00      1406
       Brute Force       0.00      0.00      0.00      1330
            Normal       1.00      0.61      0.76     58833
Unknown/Suspicious       0.00      0.00      0.00         0
        Web Attack       0.00      0.00      0.00       689

          accuracy                           0.58     62258
         macro avg       0.20      0.12      0.15     62258
      weighted avg       0.94      0.58      0.72     62258


Unseen-class detection:
Unseen labels: ['DoS/DDoS', 'Other: infilteration']
Unseen-label recall: 0.5992087440488165
Known rows incorrectly flagged unknown: 0.4208937004079797

XGBoost — CSE-CIC-IDS2018 results
Test rows: 77171
Known-label rows: 62258
Unseen-label rows: 14913

Known-class performance:
                    p

## 12. Adaptation after a newly identified attack

This simulates an analyst reviewing the separate CSE-CIC-IDS2018 adaptation batch and confirming its DoS/DDoS labels. In a live NIDS, never train directly on the model's unverified Unknown/Suspicious predictions. Keep the external test rows untouched. Retrain a candidate classifier, validate it on known IDS2025 validation rows plus CSE adaptation-validation rows, and promote only if old-class performance stays acceptable and the new class is learned. The benign-only autoencoder remains trained on Normal flows.

In [ ]:
from sklearn.metrics import recall_score

# The current source model has not seen DoS/DDoS. Use reviewed target labels
# as adaptation examples; these rows are not part of cse_external_test.
X_adapt = cse_adapt_train_df[MODEL_FEATURES].copy()
y_adapt = cse_adapt_train_df['Label'].reset_index(drop=True)
X_candidate_train = pd.concat([X_train.reset_index(drop=True), X_adapt.reset_index(drop=True)], ignore_index=True)
y_candidate_train = pd.concat([y_train.reset_index(drop=True), y_adapt], ignore_index=True)

candidate = Pipeline([
    ('preprocess', clone(preprocess)),
    ('model', RandomForestClassifier(
        n_estimators=400, class_weight='balanced_subsample',
        min_samples_leaf=2, n_jobs=-1, random_state=SEED,
    )),
])
candidate.fit(X_candidate_train, y_candidate_train)

base_known_f1 = f1_score(y_val, rf.predict(X_val), average='macro', zero_division=0)
candidate_known_f1 = f1_score(y_val, candidate.predict(X_val), average='macro', zero_division=0)
X_adapt_val = cse_adapt_val_df[MODEL_FEATURES]
y_adapt_val = cse_adapt_val_df['Label'].to_numpy()
adapt_val_pred = candidate.predict(X_adapt_val)
new_attack_recall = np.mean(adapt_val_pred == y_adapt_val)

print('Baseline known validation macro-F1:', base_known_f1)
print('Candidate known validation macro-F1:', candidate_known_f1)
print('Candidate DoS/DDoS adaptation-validation recall:', new_attack_recall)

# Example gates; agree numeric targets with the supervisor before using.
MAX_KNOWN_F1_DROP = 0.02
MIN_NEW_ATTACK_RECALL = 0.80
promote = (
    candidate_known_f1 >= base_known_f1 - MAX_KNOWN_F1_DROP
    and new_attack_recall >= MIN_NEW_ATTACK_RECALL
)
print('Promotion gate passed:', promote)

if promote:
    # Evaluate only on untouched CSE target test rows.
    cse_after_pred = candidate.predict(X_cse_test)
    print(classification_report(y_cse_test, cse_after_pred, zero_division=0))
    # Save the full pipeline, which includes preprocessing and the classifier.
    import joblib
    MODEL_DIR = Path('/content/drive/MyDrive/NIDS/model_versions')
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    artifact = {
        'version': 'rf_after_cse_dos_ddos_review_v1',
        'pipeline': candidate,
        'features': MODEL_FEATURES,
        'new_class': 'DoS/DDoS',
        'validation_macro_f1_known': candidate_known_f1,
        'validation_recall_new_class': new_attack_recall,
    }
    joblib.dump(artifact, MODEL_DIR / 'rf_after_cse_dos_ddos_review_v1.joblib')
    print('Saved candidate version; retain the previous model for rollback.')
else:
    print('Candidate rejected; keep the current model active and inspect the validation errors.')

Baseline known validation macro-F1: 0.9725965637872962
Candidate known validation macro-F1: 0.8333707596257428
Candidate DoS/DDoS adaptation-validation recall: 0.9984101748807631
Promotion gate passed: False
Candidate rejected; keep the current model active and inspect the validation errors.


## 13. Does training on both datasets improve results?

It might improve target-domain performance because the model sees more traffic variation, but it is not guaranteed. Run it as a separate pooled/adapted experiment, using labeled CSE-CIC-IDS2018 training rows and testing on the untouched CSE target set. Compare it with the zero-shot IDS2025-to-IDS2018 result above. Do not call pooled training a pure cross-dataset result: the target dataset contributed training examples.

The correct conclusion comes from the measured comparison. If scores drop on CSE-CIC-IDS2018, that is still useful evidence of dataset shift. Later, investigate drift/adaptation separately using ordered streams; a cross-dataset difference alone is not proof of temporal concept drift.

In [ ]:
# Optional pooled-training comparison after the zero-shot result is recorded.
# Train on IDS2025 known classes plus the separate labeled CSE target-domain pool.
X_cse_pool = cse_train_pool[MODEL_FEATURES].copy()
y_cse_pool = cse_train_pool['Label'].reset_index(drop=True)
X_pooled_train = pd.concat([X_train.reset_index(drop=True), X_cse_pool.reset_index(drop=True)], ignore_index=True)
y_pooled_train = pd.concat([y_train.reset_index(drop=True), y_cse_pool], ignore_index=True)

pooled_candidate = Pipeline([
    ('preprocess', clone(preprocess)),
    ('model', RandomForestClassifier(
        n_estimators=400, class_weight='balanced_subsample',
        min_samples_leaf=2, n_jobs=-1, random_state=SEED,
    )),
])
pooled_candidate.fit(X_pooled_train, y_pooled_train)
pooled_pred = pooled_candidate.predict(X_cse_test)
print('Pooled model: trained on IDS2025 plus labeled CSE training pool.')
print('This is not a zero-shot cross-dataset score.')
print(classification_report(y_cse_test, pooled_pred, zero_division=0))

Pooled model: trained on IDS2025 plus labeled CSE training pool.
This is not a zero-shot cross-dataset score.
                      precision    recall  f1-score   support

              Botnet       1.00      0.99      0.99      1406
         Brute Force       0.99      0.98      0.99      1330
            DoS/DDoS       1.00      1.00      1.00     12566
              Normal       0.97      0.98      0.97     58833
Other: infilteration       0.29      0.19      0.23      2347
            PortScan       0.00      0.00      0.00         0
          Web Attack       0.91      0.88      0.90       689

            accuracy                           0.96     77171
           macro avg       0.74      0.72      0.73     77171
        weighted avg       0.95      0.96      0.95     77171



## 14. Results to record

Record the dataset version, files sampled, sample-per-file limit, feature count before/after reduction, source and target label counts, model thresholds, per-class precision/recall/F1, macro-F1, benign false-alarm rate, held-out DoS/DDoS recall, and the exact train/validation/test role for every result.

Keep these results separate: IDS2025 in-domain; IDS2025-to-CSE-CIC-IDS2018 zero-shot; and CSE-labeled pooled/adaptation. Do not compare accuracy alone when the class distributions differ.

In [ ]:
from sklearn.base import clone
from sklearn.metrics import f1_score, classification_report
import joblib

NEW_LABEL = "DoS/DDoS"

# These adaptation rows should represent analyst-confirmed labels.
assert cse_adapt_train_df["Label"].eq(NEW_LABEL).all()
assert cse_adapt_val_df["Label"].eq(NEW_LABEL).all()

X_adapt = cse_adapt_train_df[MODEL_FEATURES].copy()
y_adapt = cse_adapt_train_df["Label"].reset_index(drop=True)

X_update_train = pd.concat(
    [X_train.reset_index(drop=True), X_adapt.reset_index(drop=True)],
    ignore_index=True,
)
y_update_train = pd.concat(
    [y_train.reset_index(drop=True), y_adapt],
    ignore_index=True,
)

candidate = Pipeline([
    ("preprocess", clone(preprocess)),
    ("model", RandomForestClassifier(
        n_estimators=400,
        class_weight="balanced_subsample",
        min_samples_leaf=2,
        n_jobs=-1,
        random_state=SEED,
    )),
])

candidate.fit(X_update_train, y_update_train)

# Evidence that DoS/DDoS has been added to the classifier's known classes.
candidate_classes = candidate.named_steps["model"].classes_
print("Candidate known classes:", candidate_classes.tolist())
assert NEW_LABEL in candidate_classes

Candidate known classes: ['Botnet', 'Brute Force', 'DoS/DDoS', 'Infiltration', 'Normal', 'PortScan', 'Web Attack']


In [ ]:
known_f1_before = f1_score(
    y_val, rf.predict(X_val), average="macro", zero_division=0
)
known_f1_after = f1_score(
    y_val, candidate.predict(X_val), average="macro", zero_division=0
)

adapt_val_pred = candidate.predict(cse_adapt_val_df[MODEL_FEATURES])
dos_recall_adapt_val = (
    adapt_val_pred == cse_adapt_val_df["Label"].to_numpy()
).mean()

print("Known-class validation macro-F1 before:", known_f1_before)
print("Known-class validation macro-F1 after:", known_f1_after)
print("DoS/DDoS adaptation-validation recall:", dos_recall_adapt_val)

Known-class validation macro-F1 before: 0.9725965637872962
Known-class validation macro-F1 after: 0.8333707596257428
DoS/DDoS adaptation-validation recall: 0.9984101748807631


In [ ]:
X_update_val = pd.concat(
    [X_val.reset_index(drop=True),
     cse_adapt_val_df[MODEL_FEATURES].reset_index(drop=True)],
    ignore_index=True,
)

val_confidence = candidate.predict_proba(X_update_val).max(axis=1)
candidate_threshold = np.quantile(val_confidence, 0.05)

test_probs = candidate.predict_proba(X_cse_test)
candidate_classes = candidate.named_steps["model"].classes_
updated_pred = candidate_classes[test_probs.argmax(axis=1)].astype(object)
updated_pred[test_probs.max(axis=1) < candidate_threshold] = "Unknown/Suspicious"

dos_test_mask = y_cse_test.eq(NEW_LABEL).to_numpy()
print("Updated model — untouched CSE test results:")
print(classification_report(y_cse_test, updated_pred, zero_division=0))

if dos_test_mask.any():
    print("DoS/DDoS correctly labelled as a known class:",
          (updated_pred[dos_test_mask] == NEW_LABEL).mean())
    print("DoS/DDoS still rejected as unknown:",
          (updated_pred[dos_test_mask] == "Unknown/Suspicious").mean())

Updated model — untouched CSE test results:
                      precision    recall  f1-score   support

              Botnet       0.00      0.00      0.00      1406
         Brute Force       0.00      0.00      0.00      1330
            DoS/DDoS       1.00      0.90      0.95     12566
              Normal       0.96      0.46      0.62     58833
Other: infilteration       0.00      0.00      0.00      2347
  Unknown/Suspicious       0.00      0.00      0.00         0
          Web Attack       0.00      0.00      0.00       689

            accuracy                           0.50     77171
           macro avg       0.28      0.19      0.22     77171
        weighted avg       0.89      0.50      0.63     77171

DoS/DDoS correctly labelled as a known class: 0.8962279166003502
DoS/DDoS still rejected as unknown: 0.10377208339964984


In [ ]:
MAX_KNOWN_F1_DROP = 0.02
MIN_NEW_CLASS_RECALL = 0.80

promote = (
    known_f1_after >= known_f1_before - MAX_KNOWN_F1_DROP
    and dos_recall_adapt_val >= MIN_NEW_CLASS_RECALL
)

print("Promotion gate passed:", promote)

if promote:
    model_dir = Path("/content/drive/MyDrive/NIDS/model_versions")
    model_dir.mkdir(parents=True, exist_ok=True)

    joblib.dump(
        {
            "version": "rf_after_dos_ddos_adaptation_v1",
            "pipeline": candidate,
            "features": MODEL_FEATURES,
            "classes": candidate_classes.tolist(),
            "unknown_threshold": candidate_threshold,
        },
        model_dir / "rf_after_dos_ddos_adaptation_v1.joblib",
    )
    print("Saved the updated model version.")
else:
    print("Do not promote yet; review the validation results and adaptation data.")

Promotion gate passed: False
Do not promote yet; review the validation results and adaptation data.
